# Securing ML Model APIs — Authentication, Rate Limiting, Input Validation

A deployed ML model is only as secure as the API in front of it. This notebook covers four concrete security layers every ML API should have:
1. API key authentication
2. JWT token authentication
3. Rate limiting
4. Input validation and sanitization

All code runs fully locally — no cloud credentials required.

## Learning Objectives

By the end of this notebook you will be able to:
1. Implement API key validation in FastAPI using a header dependency
2. Issue and validate JWT tokens and protect an endpoint that requires one
3. Build in-memory rate limiting to prevent API abuse
4. Write an input sanitization function and explain why it matters for ML APIs

## Setup

In [1]:
# WHAT: install the security toolkit — FastAPI, httpx, and python-jose for JWTs.
# WHY: every defense in this lesson (API keys, tokens, rate limits) is built and
# TESTED live, so the libraries must be present up front.
import subprocess, sys

packages = ['fastapi', 'httpx', 'python-jose[cryptography]']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   capture_output=True)
print('Dependencies installed: fastapi, httpx, python-jose')

Dependencies installed: fastapi, httpx, python-jose


## Section 1 — API Key Authentication

The simplest auth mechanism: the caller includes a secret key in a request header. The server validates it before processing the request.

**Authentication** = proving who you are.
**Authorization** = deciding what you are allowed to do.

An API key handles authentication. A role or permission system handles authorization.

In [2]:
# WHAT: protect a /predict endpoint with an API key checked by a FastAPI dependency.
# WHY: API keys are the simplest auth layer — note the two distinct failures:
# 401 when the key is MISSING vs 403 when it is present but WRONG.
import secrets
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import APIKeyHeader
from fastapi.testclient import TestClient

# In production: store this in an environment variable, not in code
VALID_API_KEYS = {
    'sk-student-key-001': 'student_alice',
    'sk-student-key-002': 'student_bob',
}

app_apikey = FastAPI(title='ML API with API Key Auth')
api_key_header = APIKeyHeader(name='X-API-Key', auto_error=False)

# The dependency runs before the endpoint: no valid key, no prediction.
def require_api_key(api_key: str = Depends(api_key_header)):
    """FastAPI dependency: validates the API key and returns the owner name."""
    if api_key is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail='Missing X-API-Key header',
        )
    if api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail='Invalid API key',
        )
    return VALID_API_KEYS[api_key]

@app_apikey.post('/predict')
def predict(caller: str = Depends(require_api_key)):
    return {'prediction': 'setosa', 'caller': caller}

# Test it
client = TestClient(app_apikey)

print('Test 1 — no API key:')
r = client.post('/predict')
print(f'  Status: {r.status_code}, Body: {r.json()}')

print('Test 2 — wrong API key:')
r = client.post('/predict', headers={'X-API-Key': 'wrong-key'})
print(f'  Status: {r.status_code}, Body: {r.json()}')

print('Test 3 — valid API key:')
r = client.post('/predict', headers={'X-API-Key': 'sk-student-key-001'})
print(f'  Status: {r.status_code}, Body: {r.json()}')

Test 1 — no API key:
  Status: 401, Body: {'detail': 'Missing X-API-Key header'}
Test 2 — wrong API key:
  Status: 403, Body: {'detail': 'Invalid API key'}
Test 3 — valid API key:
  Status: 200, Body: {'prediction': 'setosa', 'caller': 'student_alice'}


/Users/abdullah/Downloads/AI Diploma/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## Section 2 — JWT Token Authentication

JSON Web Tokens (JWT) are a more sophisticated auth mechanism. A JWT contains:
- **Header**: algorithm used to sign the token
- **Payload**: claims about the user (user ID, expiry, roles)
- **Signature**: proves the token was issued by the server and hasn't been tampered with

The format is: `base64(header).base64(payload).signature`

Common use: a `/login` endpoint issues a short-lived JWT (15 min to 1 hour). The client sends it on every subsequent request in the `Authorization: Bearer <token>` header.

In [3]:
# WHAT: build JWT helpers — issue a signed token, then validate and decode it.
# WHY: unlike API keys, JWTs carry claims (who, until when) and are verified by
# SIGNATURE, so the server does not need a lookup table of issued tokens.
from datetime import datetime, timedelta, timezone
from jose import jwt, JWTError
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.testclient import TestClient
from pydantic import BaseModel

SECRET_KEY = 'aiat125-super-secret-replace-in-production'
ALGORITHM = 'HS256'
TOKEN_EXPIRE_MINUTES = 30

def create_access_token(subject: str) -> str:
    """Issue a JWT that expires in TOKEN_EXPIRE_MINUTES."""
    expire = datetime.now(timezone.utc) + timedelta(minutes=TOKEN_EXPIRE_MINUTES)
    payload = {'sub': subject, 'exp': expire}
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def decode_access_token(token: str) -> str:
    """Validate a JWT and return the subject claim."""
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        subject = payload.get('sub')
        if subject is None:
            raise ValueError('Token missing sub claim')
        return subject
    except JWTError as e:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail=f'Invalid token: {e}',
            headers={'WWW-Authenticate': 'Bearer'},
        )

# A JWT is three base64 parts: header.payload.signature — decode and look inside.
# Show the token structure
token = create_access_token('user123')
print(f'Issued JWT: {token[:60]}...')

# Decode and inspect
import base64, json
header_b64, payload_b64, sig = token.split('.')
# Pad base64 string
header_json = json.loads(base64.b64decode(header_b64 + '=='))
payload_json = json.loads(base64.b64decode(payload_b64 + '=='))
print(f'\nHeader  : {header_json}')
print(f'Payload : {payload_json}')
print(f'Expires : {datetime.fromtimestamp(payload_json["exp"], tz=timezone.utc).isoformat()}')

Issued JWT: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJzdWIiOiJ1c2VyMTIzIiw...

Header  : {'alg': 'HS256', 'typ': 'JWT'}
Payload : {'sub': 'user123', 'exp': 1787505787}
Expires : 2026-08-23T17:23:07+00:00


In [4]:
# WHAT: wire the JWT flow into an API: /login issues tokens, /predict requires them.
# WHY: this is the standard two-step dance — exchange credentials for a token
# once, then present the token on every call; tampering breaks the signature.
app_jwt = FastAPI(title='ML API with JWT Auth')
bearer_scheme = HTTPBearer(auto_error=False)

class LoginRequest(BaseModel):
    username: str
    password: str

FAKE_USER_DB = {'alice': 'password123', 'bob': 'hunter2'}

@app_jwt.post('/login')
def login(body: LoginRequest):
    if FAKE_USER_DB.get(body.username) != body.password:
        raise HTTPException(status_code=401, detail='Bad credentials')
    token = create_access_token(subject=body.username)
    return {'access_token': token, 'token_type': 'bearer'}

def require_jwt(credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme)):
    if credentials is None:
        raise HTTPException(status_code=401, detail='Missing Authorization header')
    return decode_access_token(credentials.credentials)

@app_jwt.post('/predict')
def protected_predict(user: str = Depends(require_jwt)):
    return {'prediction': 'versicolor', 'user': user}

# Four moments of the token lifecycle: issue, reject-missing, accept, reject-tampered.
# Test the full flow
client = TestClient(app_jwt)

print('Step 1 — login to get a token:')
r = client.post('/login', json={'username': 'alice', 'password': 'password123'})
print(f'  Status: {r.status_code}')
jwt_token = r.json()['access_token']
print(f'  Token (first 40 chars): {jwt_token[:40]}...')

print('\nStep 2 — call the protected endpoint without token:')
r = client.post('/predict')
print(f'  Status: {r.status_code}, Body: {r.json()}')

print('\nStep 3 — call with valid token:')
r = client.post('/predict', headers={'Authorization': f'Bearer {jwt_token}'})
print(f'  Status: {r.status_code}, Body: {r.json()}')

print('\nStep 4 — call with tampered token:')
bad_token = jwt_token[:-5] + 'XXXXX'
r = client.post('/predict', headers={'Authorization': f'Bearer {bad_token}'})
print(f'  Status: {r.status_code}, Body: {r.json()}')

Step 1 — login to get a token:
  Status: 200
  Token (first 40 chars): eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ...

Step 2 — call the protected endpoint without token:
  Status: 401, Body: {'detail': 'Missing Authorization header'}

Step 3 — call with valid token:
  Status: 200, Body: {'prediction': 'versicolor', 'user': 'alice'}

Step 4 — call with tampered token:
  Status: 401, Body: {'detail': 'Invalid token: Signature verification failed.'}


## Section 3 — Rate Limiting

Rate limiting prevents a single client from flooding your API. Without it, one bad actor can cause your endpoint to hit its resource limits, degrading service for everyone else.

A simple approach: keep a dictionary of `{ip_address: [request_timestamps]}` and reject requests if the count in the last 60 seconds exceeds the limit.

In [5]:
# WHAT: implement a sliding-window rate limiter in ~15 lines of Python.
# WHY: without limits, one buggy or malicious client can starve everyone else —
# rate limiting is availability protection, per client, not just security.
import time
from collections import defaultdict
from functools import wraps

# In-memory rate limiter: {client_id: [timestamps of recent requests]}
request_log = defaultdict(list)

RATE_LIMIT = 5         # max requests per window
WINDOW_SECONDS = 10    # sliding window size

# Keep only timestamps inside the window, then check if the budget is spent.
def is_rate_limited(client_id: str) -> bool:
    """Return True if the client has exceeded the rate limit."""
    now = time.time()
    # Remove timestamps older than the window
    request_log[client_id] = [
        t for t in request_log[client_id]
        if now - t < WINDOW_SECONDS
    ]
    if len(request_log[client_id]) >= RATE_LIMIT:
        return True
    request_log[client_id].append(now)
    return False

# Requests 6-8 from the same IP should be rejected; a new IP starts fresh.
# Simulate requests from two clients
print(f'Rate limit: {RATE_LIMIT} requests per {WINDOW_SECONDS} seconds')
print()

for i in range(8):
    client_id = '192.168.1.1'
    limited = is_rate_limited(client_id)
    verdict = 'REJECTED (429 Too Many Requests)' if limited else 'ACCEPTED (200 OK)'
    print(f'Request {i+1:02d} from {client_id}: {verdict}')

print()
print('Different client (different IP, not rate-limited):')
for i in range(3):
    limited = is_rate_limited('10.0.0.5')
    verdict = 'REJECTED' if limited else 'ACCEPTED'
    print(f'  Request {i+1}: {verdict}')

Rate limit: 5 requests per 10 seconds

Request 01 from 192.168.1.1: ACCEPTED (200 OK)
Request 02 from 192.168.1.1: ACCEPTED (200 OK)
Request 03 from 192.168.1.1: ACCEPTED (200 OK)
Request 04 from 192.168.1.1: ACCEPTED (200 OK)
Request 05 from 192.168.1.1: ACCEPTED (200 OK)
Request 06 from 192.168.1.1: REJECTED (429 Too Many Requests)
Request 07 from 192.168.1.1: REJECTED (429 Too Many Requests)
Request 08 from 192.168.1.1: REJECTED (429 Too Many Requests)

Different client (different IP, not rate-limited):
  Request 1: ACCEPTED
  Request 2: ACCEPTED
  Request 3: ACCEPTED


In [6]:
# Integrate rate limiting into FastAPI as a dependency
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

request_log.clear()  # reset for this demo

app_rl = FastAPI(title='ML API with Rate Limiting')

# Same limiter as above, packaged as a dependency with a proper 429 + Retry-After.
def check_rate_limit(request: Request):
    """FastAPI dependency: raise 429 if client exceeds rate limit."""
    client_ip = request.client.host if request.client else 'testclient'
    if is_rate_limited(client_ip):
        raise HTTPException(
            status_code=status.HTTP_429_TOO_MANY_REQUESTS,
            detail=f'Rate limit exceeded: max {RATE_LIMIT} requests per {WINDOW_SECONDS}s',
            headers={'Retry-After': str(WINDOW_SECONDS)},
        )

@app_rl.post('/predict')
def predict_with_rl(_: None = Depends(check_rate_limit)):
    return {'prediction': 'virginica'}

client = TestClient(app_rl)
print(f'Sending 8 rapid requests (limit is {RATE_LIMIT}):')
for i in range(8):
    r = client.post('/predict')
    print(f'  Request {i+1}: HTTP {r.status_code}')

Sending 8 rapid requests (limit is 5):
  Request 1: HTTP 200
  Request 2: HTTP 200
  Request 3: HTTP 200
  Request 4: HTTP 200
  Request 5: HTTP 200
  Request 6: HTTP 429
  Request 7: HTTP 429
  Request 8: HTTP 429


## Section 4 — Input Sanitization

Even if a caller is authenticated, you must validate their input before passing it to the model. Reasons:

- **Crashes**: passing a string where a float is expected will throw a `ValueError` inside your model
- **Adversarial inputs**: extreme values or NaN can produce garbage predictions without crashing
- **Security**: malformed or extreme values are a common probing technique. Validating shape, dtype, and range blocks the easy abuse cases. (Note: sophisticated attacks such as model-inversion or membership-inference, which try to leak training data, need defenses *beyond* input validation — validation alone does not stop them.)

Always validate: shape, dtype, value range, and presence of NaN/Inf.

In [7]:
# WHAT: sanitize model input — dtype, shape, NaN/Inf, and physical value ranges.
# WHY: schema validation alone accepts sepal_length=99.0; range checks encode
# DOMAIN knowledge, the difference between valid JSON and valid data.
import numpy as np

# Feature constraints for the iris model
FEATURE_RANGES = {
    'sepal_length': (4.0, 8.0),
    'sepal_width':  (1.5, 5.0),
    'petal_length': (0.5, 7.5),
    'petal_width':  (0.0, 3.0),
}
FEATURE_NAMES = list(FEATURE_RANGES.keys())

# Four layers, cheapest first: convert, shape-check, finiteness, then ranges.
def sanitize_input(raw_features):
    """
    Validate and sanitize model input.
    Returns a clean numpy array or raises ValueError.
    """
    # 1. Convert to numpy array
    try:
        arr = np.array(raw_features, dtype=float)
    except (ValueError, TypeError) as e:
        raise ValueError(f'Cannot convert input to float array: {e}')

    # 2. Check shape
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    if arr.shape[1] != len(FEATURE_NAMES):
        raise ValueError(
            f'Expected {len(FEATURE_NAMES)} features, got {arr.shape[1]}'
        )

    # 3. Check for NaN or Infinity
    if not np.isfinite(arr).all():
        raise ValueError('Input contains NaN or Infinity values')

    # 4. Check value ranges
    for col_idx, (name, (low, high)) in enumerate(FEATURE_RANGES.items()):
        col = arr[:, col_idx]
        if np.any(col < low) or np.any(col > high):
            raise ValueError(
                f'{name} out of range [{low}, {high}]: got {col.tolist()}'
            )

    return arr

# Every malformed case must be BLOCKED with a message naming the exact problem.
# Test with various inputs
test_cases = [
    ([5.1, 3.5, 1.4, 0.2],          'Valid input'),
    ([5.1, 3.5, 1.4],               'Wrong number of features'),
    ([5.1, 3.5, float('nan'), 0.2], 'NaN value'),
    ([5.1, 3.5, float('inf'), 0.2], 'Infinity value'),
    ([99.0, 3.5, 1.4, 0.2],        'Extreme value (sepal_length=99)'),
    (['a', 3.5, 1.4, 0.2],         'String in numeric array'),
]

for features, description in test_cases:
    try:
        clean = sanitize_input(features)
        print(f'  PASS [{description}]: shape={clean.shape}')
    except ValueError as e:
        print(f'  BLOCKED [{description}]: {e}')

  PASS [Valid input]: shape=(1, 4)
  BLOCKED [Wrong number of features]: Expected 4 features, got 3
  BLOCKED [NaN value]: Input contains NaN or Infinity values
  BLOCKED [Infinity value]: Input contains NaN or Infinity values
  BLOCKED [Extreme value (sepal_length=99)]: sepal_length out of range [4.0, 8.0]: got [99.0]
  BLOCKED [String in numeric array]: Cannot convert input to float array: could not convert string to float: 'a'


## Section 5 — HTTPS in Production

Never serve an ML API over plain HTTP in production. Here is why:

- **API keys in transit**: if you use API key auth over HTTP, any network observer can capture the key and replay your requests
- **JWT tokens in transit**: same problem — the token is the credential
- **Model outputs**: predictions can be business-sensitive (pricing models, fraud scores, medical diagnosis)

How HTTPS works in a typical cloud deployment:

```
Client
  |-- HTTPS (TLS encrypted) -->
                          Load Balancer / API Gateway
                            (terminates TLS, holds certificate)
                              |-- HTTP (internal network, trusted) -->
                                             Your FastAPI/Flask service
```

You do not configure TLS in FastAPI itself. The cloud provider's load balancer (AWS ALB, GCP Load Balancer, Azure Application Gateway) handles TLS termination and forwards decrypted traffic to your service over the private internal network.

For local development, HTTPS is not needed. For any public-facing service, it is mandatory.

## Section 6 — Putting It Together

A production ML API applies all four layers simultaneously.

In [8]:
# WHAT: assemble the full secure API — API key auth + rate limiting + input
# sanitization + the real model, all in one endpoint.
# WHY: defenses compose in order: who are you (403), are you flooding (429),
# is the data sane (422) — only then does the model run (200).
import joblib
from fastapi import FastAPI, Depends, Request, HTTPException, status
from fastapi.security import APIKeyHeader
from fastapi.testclient import TestClient
from pydantic import BaseModel
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Train a model
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
clf = RandomForestClassifier(n_estimators=50, random_state=42).fit(scaler.transform(X_train), y_train)

# Security config
PROD_API_KEYS = {'sk-prod-key-abc': 'service_a', 'sk-prod-key-xyz': 'service_b'}
api_key_header = APIKeyHeader(name='X-API-Key', auto_error=False)
rl_log = defaultdict(list)
PROD_RATE_LIMIT = 100
PROD_WINDOW = 60

def validate_key(api_key: str = Depends(api_key_header)):
    if not api_key or api_key not in PROD_API_KEYS:
        raise HTTPException(status_code=403, detail='Invalid or missing API key')
    return PROD_API_KEYS[api_key]

def check_rl(request: Request):
    ip = request.client.host if request.client else 'testclient'
    now = time.time()
    rl_log[ip] = [t for t in rl_log[ip] if now - t < PROD_WINDOW]
    if len(rl_log[ip]) >= PROD_RATE_LIMIT:
        raise HTTPException(status_code=429, detail='Rate limit exceeded')
    rl_log[ip].append(now)

class PredictRequest(BaseModel):
    features: list

app_secure = FastAPI(title='Secure ML API')

# The endpoint declares its guards as dependencies; the body is pure inference.
@app_secure.post('/predict')
def secure_predict(
    body: PredictRequest,
    caller: str = Depends(validate_key),
    _: None = Depends(check_rl),
):
    try:
        clean = sanitize_input(body.features)
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

    scaled = scaler.transform(clean)
    pred = int(clf.predict(scaled)[0])
    conf = float(clf.predict_proba(scaled)[0].max())
    return {
        'prediction': iris.target_names[pred],
        'confidence': round(conf, 3),
        'caller': caller,
    }

# One request per defense layer, showing which status code each layer owns.
# Run tests
client = TestClient(app_secure)
valid_features = [5.1, 3.5, 1.4, 0.2]

print('Secure API tests:')
tests = [
    ('No API key',         {},                                     {'features': valid_features}),
    ('Bad API key',        {'X-API-Key': 'bad'},                   {'features': valid_features}),
    ('Valid key, bad input', {'X-API-Key': 'sk-prod-key-abc'},     {'features': [99, 3.5, 1.4, 0.2]}),
    ('Valid key, valid input', {'X-API-Key': 'sk-prod-key-abc'},   {'features': valid_features}),
]

for label, headers, body in tests:
    r = client.post('/predict', headers=headers, json=body)
    print(f'  [{label}] HTTP {r.status_code}: {r.json()}')

Secure API tests:
  [No API key] HTTP 403: {'detail': 'Invalid or missing API key'}
  [Bad API key] HTTP 403: {'detail': 'Invalid or missing API key'}
  [Valid key, bad input] HTTP 422: {'detail': 'sepal_length out of range [4.0, 8.0]: got [99.0]'}
  [Valid key, valid input] HTTP 200: {'prediction': 'setosa', 'confidence': 1.0, 'caller': 'service_a'}


## Summary

Four security layers for an ML API:

| Layer | Tool | What it prevents |
|---|---|---|
| **Authentication** | API key (`X-API-Key`) or JWT (`Authorization: Bearer`) | Unauthorized callers |
| **Rate limiting** | In-memory sliding window, or cloud API Gateway | Abuse, DDoS, runaway costs |
| **Input validation** | `sanitize_input()` before calling the model | Crashes, garbage predictions, adversarial inputs |
| **HTTPS** | Cloud load balancer with TLS certificate | Credential interception, eavesdropping |

Apply all four. They are complementary, not alternatives to each other.

## Self-Check

1. **What is the difference between authentication and authorization?**
   *(Give a one-sentence definition of each. Can something be authenticated but not authorized?)*

2. **What does JWT stand for and what is in the payload?**
   *(Name the full term and at least two fields you would typically find in the payload.)*

3. **Why should you validate model inputs even if your API is authenticated?**
   *(Think about what an authenticated but malicious caller — or simply a buggy caller — could send.)*

## 📚 References

1. Goodfellow, I. J., Shlens, J., & Szegedy, C. (2015). *Explaining and Harnessing Adversarial Examples*. ICLR. <https://arxiv.org/abs/1412.6572>
2. Papernot, N., McDaniel, P., Sinha, A., & Wellman, M. (2016). *Towards the Science of Security and Privacy in Machine Learning*. arXiv. <https://arxiv.org/abs/1611.03814>
3. Kumar, R. S. S., Nyström, M., Lambert, J., et al. (2020). *Adversarial Machine Learning — Industry Perspectives*. IEEE S&P Workshops. <https://arxiv.org/abs/2002.05646>
